## Set up for using Claude Code in Google Colab environment.
https://note.com/sunwood_ai_labs/n/ne02f78220650

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from google.colab import userdata
import os

print("loading authentication")

GH_TOKEN = userdata.get("GH_TOKEN")
GIT_USER_NAME = userdata.get("GIT_USER_NAME")
GIT_USER_EMAIL = userdata.get("GIT_USER_EMAIL")

missing = []
if not GH_TOKEN:
    missing.append("GH_TOKEN")
if not GIT_USER_NAME:
    missing.append("GIT_USER_NAME")
if not GIT_USER_EMAIL:
    missing.append("GIT_USER_EMAIL")

if missing:
    raise RuntimeError(
        f"the following secrets are missing: {', '.join(missing)}",
        "Register the necessary information in 🔑(Secrets) and turn on the Notebook access"
    )

print("authentication loaded")

loading authentication
authentication loaded


In [3]:
os.environ["GH_TOKEN"] = GH_TOKEN
os.environ["GIT_USER_NAME"] = GIT_USER_NAME
os.environ["GIT_USER_EMAIL"] = GIT_USER_EMAIL

In [4]:
!git config --global user.name "$GIT_USER_NAME"
!git config --global user.email "$GIT_USER_EMAIL"

print("Setting of the git owner completed")
print(f"  user.name : {GIT_USER_NAME}")
print(f"  user.email: {GIT_USER_EMAIL}")

Setting of the git owner completed
  user.name : yoji-toriumi
  user.email: yoji.toriumi@gmail.com


In [5]:
%%bash
set -euo pipefail

TOKEN="${GH_TOKEN:-}"
if [ -z "$TOKEN" ]; then
  echo "ERROR: GH_TOKEN is blank."
  exit 1
fi

unset GH_TOKEN GITHUB_TOKEN GH_ENTERPRISE_TOKEN

printf "%s" "$TOKEN" | gh auth login --hostname github.com --git-protocol https --with-token

gh auth setup-git

echo "✅ gh authentication status:"
gh auth status

echo "✅ Setting files:"
ls -la ~/.config/gh || true

✅ gh authentication status:
github.com
  ✓ Logged in to github.com account yoji-toriumi (/root/.config/gh/hosts.yml)
  - Active account: true
  - Git operations protocol: https
  - Token: ghp_************************************
  - Token scopes: 'admin:org', 'repo'
✅ Setting files:
total 16
drwxr-x--x 2 root root 4096 Jan 15 17:52 .
drwxr-xr-x 1 root root 4096 Jan 15 17:52 ..
-rw------- 1 root root 1660 Jan 15 17:52 config.yml
-rw------- 1 root root  216 Jan 15 17:52 hosts.yml


In [6]:
%%bash
set -euo pipefail

hr()   { printf "\n━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n"; }
step() { hr; echo "✅ Step $1/$TOTAL：$2"; }
info() { echo "  ℹ️  $*"; }
ok()   { echo "  🟢 $*"; }
warn() { echo "  🟡 $*"; }

TOTAL=6

step 1 "Confirm encironment setting"
info "USER=${USER:-unknown}"
info "HOME=${HOME:-unknown}"
info "SHELL=${SHELL:-unknown}"
info "PATH=${PATH:-unknown}"

step 2 "Confirm installation of Claude Code"
if [ -x "$HOME/.local/bin/claude" ]; then
  ok "Found: $HOME/.local/bin/claude"
  "$HOME/.local/bin/claude" --version || true
else
  warn "Not installed: $HOME/.local/bin/claude"
fi

step 3 "Install Claude Code if it's not installed（curl | bash）"
if [ ! -x "$HOME/.local/bin/claude" ]; then
  info "Under installation（take some time）"
  time curl -fsSL https://claude.ai/install.sh | bash
  ok "Installation completed"
else
  ok "skip the step as it's already installed"
fi

step 4 "link to /usr/local/bin/claude"
ln -sf "$HOME/.local/bin/claude" /usr/local/bin/claude
info "link status: $(ls -l /usr/local/bin/claude)"
ok "version: $(/usr/local/bin/claude --version)"

step 5 "cc の退避（必要なら）＋ cc（通常モード）を作成"
if command -v cc >/dev/null 2>&1; then
  ORIG="$(command -v cc)"
  ORIG_REAL="$(readlink -f "$ORIG" || true)"
  info "現在の cc: $ORIG（実体: ${ORIG_REAL:-unknown}）"
  if [ "$ORIG" != "/usr/local/bin/cc" ] && [ -n "${ORIG_REAL:-}" ]; then
    ln -sf "$ORIG_REAL" /usr/local/bin/cc-gcc
    ok "元の cc を退避: /usr/local/bin/cc-gcc -> $ORIG_REAL"
  else
    ok "退避は不要"
  fi
fi

cat >/usr/local/bin/cc <<'EOF'
#!/usr/bin/env bash
set -euo pipefail
exec /usr/local/bin/claude "$@"
EOF
chmod +x /usr/local/bin/cc
info "cc: $(ls -l /usr/local/bin/cc)"

step 6 "最終確認"
info "which claude: $(command -v claude || true)"
info "which cc:     $(command -v cc || true)"
ok "cc --version:  $(cc --version)"

hr
ok "完了：ターミナルで 'cc' または 'claude' を実行できます。"
warn "注意：Cコンパイル用途の cc が必要なら 'cc-gcc' を使用（退避されている場合）。"


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
✅ Step 1/6：Confirm encironment setting
  ℹ️  USER=unknown
  ℹ️  HOME=/root
  ℹ️  SHELL=/bin/bash
  ℹ️  PATH=/opt/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
✅ Step 2/6：Confirm installation of Claude Code
  🟡 Not installed: /root/.local/bin/claude

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
✅ Step 3/6：Install Claude Code if it's not installed（curl | bash）
  ℹ️  Under installation（take some time）
Setting up Claude Code...

✔ Claude Code successfully installed!

  Version: 2.1.7

  Location: ~/.local/bin/claude


  Next: Run claude --help to get started

⚠ Setup notes:
  • Native installation exists but ~/.local/bin is not in your PATH. Run:

  echo 'export PATH="$HOME/.local/bin:$PATH"' >> ~/.bashrc && source ~/.bashrc


✅ Installation complete!

  🟢 Installation completed

━━━━


real	0m22.376s
user	0m10.097s
sys	0m3.262s


In [7]:
%cd /content
print("✅ 作業ディレクトリ: /content")
print("\n🚀 準備完了！ターミナルを開いて claude を起動してください")

/content
✅ 作業ディレクトリ: /content

🚀 準備完了！ターミナルを開いて claude を起動してください


In [8]:
cd

/root


In [12]:
!streamlit run
/content/drive/MyDrive/Municipal_Credit_Assessment/app.py
--server.port 8501

SyntaxError: invalid syntax (ipython-input-2336044079.py, line 3)

In [13]:
!streamlit run
  /content/drive/MyDrive/Municipal_Credit_Assessment/app.py
   --server.port 8501

IndentationError: unexpected indent (ipython-input-1771666171.py, line 2)